# NILM – Semaine 7 : préparer le modèle pour la carte STM32 (NUCLEO-N657X0-Q)

**Objectif.** Transformer le réseau retenu en semaine 6 (run `ukdale_augoff_s0`) en un fichier que les outils de ST savent compiler pour la carte NUCLEO-N657X0-Q (microcontrôleur STM32N6 avec accélérateur Neural-ART), **sans perdre en qualité**.

Le notebook fait 4 choses, dans l'ordre :
1. il **recharge** le modèle et vérifie qu'il redonne exactement les scores de la semaine 6 ;
2. il simule la **fenêtre glissante** de la carte : le réseau ne verra pas 3 mois d'un coup, mais une fenêtre de 272 minutes toutes les 15 minutes ; je vérifie que cela donne les mêmes scores ;
3. il **exporte** le réseau au format ONNX (format d'échange lu par ST Edge AI), puis le **compresse en entiers 8 bits** (int8 : 4 fois plus petit et bien plus rapide sur l'accélérateur de la N6) ;
4. il **compare** les scores (modèle d'origine / fenêtre glissante / int8), puis écrit tous les fichiers nécessaires au programme de la carte.

Tout est écrit dans le dossier `export_s7/`.

## 0. Installation (une seule fois)
Trois bibliothèques s'ajoutent à l'environnement des semaines précédentes : `onnx` (le format), `onnxruntime` (pour exécuter et compresser le fichier ONNX sur le Mac) et `onnxscript` (utilisé par l'exportateur de PyTorch). Après l'installation, il faut redémarrer le noyau.

In [ ]:
%pip install -q onnx onnxruntime onnxscript

## 1. Réglages
- `DATA_DIR` : les mêmes données UK-DALE qu'en semaine 6 ;
- `RUN_DIR` : le run de référence `runs_s6/ukdale_augoff_s0`, dont je recharge les poids ;
- `EXPORT_DIR` : le dossier de sortie `export_s7/` ;
- réglages du modèle (`CH`, `DILS`, seuils, découpe) : **identiques à l'entraînement**, ils ne doivent pas changer ;
- réglages de la carte : fenêtre de **272 minutes** (`WIN`), un calcul toutes les **15 minutes** (`STEP`), contexte de **127 minutes** (`RF`, le champ réceptif du réseau d'un côté), et nombre de fenêtres de calibration pour la compression (`N_CALIB`).

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort

# ================== CHEMINS (les mêmes qu'en semaine 6) ==================
DATA_DIR = Path.home() / "Desktop/MASTER/Projet/Modele_IA/house_1/an1"
RUN_DIR = Path("runs_s6/ukdale_augoff_s0")          # le run de référence (meilleur sur le réel)
EXPORT_DIR = Path("export_s7"); EXPORT_DIR.mkdir(exist_ok=True)
# ================== RÉGLAGES DU MODÈLE (ne pas changer : identiques à l'entraînement) ==================
MAINS = "aggregate"
APPLIANCES = ["fridge", "kettle", "washing_machine", "dishwasher"]
LOAD = ("2013-03-01", "2014-02-28")
RESAMPLE = "1min"
ON_W = {"fridge": 30, "kettle": 500, "washing_machine": 20, "dishwasher": 20}
TEST_FRAC, VAL_FRAC = 0.30, 0.07
CH = 32
DILS = (1, 2, 4, 8, 16, 32, 64)
# ================== RÉGLAGES DE LA CARTE ==================
RF = sum(DILS)            # 127 : le réseau regarde 127 minutes avant et 127 minutes après chaque minute
STEP = 15                 # un calcul (et un message) toutes les 15 minutes
WIN = 272                 # fenêtre vue par la carte : 127 + 15 + 127 = 269, arrondi à 272 (multiple de 16)
N_CALIB = 500             # fenêtres d'entraînement utilisées pour régler la compression int8
torch.manual_seed(0); np.random.seed(0)
print("onnx", onnx.__version__, "| onnxruntime", ort.__version__, "| torch", torch.__version__)
print("retard des résultats sur la carte : entre", RF, "et", RF + STEP, "minutes (environ 2 h à 2 h 20)")

## 2. Données UK-DALE (exactement comme en semaine 6)
Je relis l'année de données, avec la même grille de temps, les mêmes minutes valides, la même découpe et les mêmes échelles. C'est indispensable pour comparer les scores au chiffre près avec la semaine 6.

In [ ]:
def read_labels(folder):
    labels = {}
    for line in (folder / "labels.dat").read_text().splitlines():
        p = line.split(maxsplit=1)
        if len(p) == 2:
            labels[p[1].strip()] = int(p[0])
    return labels

def read_channel(folder, ch):
    df = pd.read_csv(folder / f"channel_{ch}.dat", sep=r"\s+", header=None, usecols=[0, 1], names=["t", "w"])
    df.index = pd.to_datetime(df["t"], unit="s")
    s = df["w"].astype(float)
    return s[~s.index.duplicated()].sort_index().loc[LOAD[0]:LOAD[1]]

labels = read_labels(DATA_DIR)
series = {n: read_channel(DATA_DIR, labels[n]).resample(RESAMPLE).mean() for n in [MAINS] + APPLIANCES}
data = pd.DataFrame(series)
data = data.loc[:min(s.index.max() for s in series.values())]
data = data.reindex(pd.date_range(data.index.min(), data.index.max(), freq=RESAMPLE))
valid = data.notna().all(axis=1).values
T = len(data)
agg = data[MAINS].fillna(0).values.astype(np.float32)
Yw = data[APPLIANCES].fillna(0).values.astype(np.float32).T
thr = np.array([ON_W[a] for a in APPLIANCES], dtype=np.float32)
n_app = len(APPLIANCES)
i_test = int(T * (1 - TEST_FRAC)); i_val = int(T * (1 - TEST_FRAC - VAL_FRAC))
tr = valid.copy(); tr[i_val:] = False
te = valid.copy(); te[:i_test] = False
scale_y = np.array([np.percentile(Yw[i, tr & (Yw[i] > thr[i])], 99) for i in range(n_app)], dtype=np.float32)
hour = (data.index.hour + data.index.minute / 60).values
X = np.vstack([agg / 1000.0, np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24)]).astype(np.float32)
X[:, ~valid] = 0.0
print(f"{T} minutes | test à partir de {data.index[i_test]} | échelles (W) :", dict(zip(APPLIANCES, scale_y.round(1).tolist())))

## 3. Recharger le réseau et vérifier qu'il redonne les scores de la semaine 6
Je reconstruis la même architecture, je charge `nilm_cnn.pt` et je recalcule les scores sur la période de test complète. Ils doivent être identiques à ceux de la semaine 6 (F1 de 0,75 à 0,91). Sur la carte, la décision « allumé » se fait simplement par sortie > 0, ce qui équivaut à une probabilité > 0,5 sans calculer d'exponentielle.

In [ ]:
class Net(nn.Module):
    def __init__(self, n_in, n_out, ch=32, dils=DILS):
        super().__init__()
        self.inp = nn.Conv1d(n_in, ch, 1)
        self.blocks = nn.ModuleList([nn.Conv1d(ch, ch, 3, padding=d, dilation=d) for d in dils])
        self.out = nn.Conv1d(ch, n_out, 1)
    def forward(self, x):
        h = F.relu(self.inp(x))
        for b in self.blocks:
            h = h + F.relu(b(h))
        return self.out(h)

net = Net(3, 2 * n_app, CH)
net.load_state_dict(torch.load(RUN_DIR / "nilm_cnn.pt", map_location="cpu"))
net.eval()

def to_pred(out):
    # out : (2*n_app, temps) -> puissance (W) et état allumé ; sur la carte : allumé <=> sortie > 0 (pas besoin d'exponentielle)
    reg = np.maximum(out[:n_app], 0) * scale_y[:, None]
    on = out[n_app:] > 0
    return reg * on, on

def scores(pred, on, nom):
    rows = []
    for i, a in enumerate(APPLIANCES):
        t, p = Yw[i, te] > thr[i], on[i, te]
        tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
        e_t, e_p = Yw[i, te].sum() / 60000, pred[i, te].sum() / 60000
        rows.append({"appareil": a, "version": nom, "F1": 2 * tp / (2 * tp + fp + fn),
                     "erreur énergie (%)": 100 * abs(e_p - e_t) / e_t})
    return pd.DataFrame(rows)

with torch.no_grad():
    out_full = net(torch.tensor(X[:, i_test - 512:T])[None])[0, :, 512:].numpy()
pred_ref = np.zeros((n_app, T), np.float32); on_ref = np.zeros((n_app, T), bool)
pred_ref[:, i_test:], on_ref[:, i_test:] = to_pred(out_full)
s_ref = scores(pred_ref, on_ref, "1. modèle d'origine (PC)")

attendu = pd.read_csv(RUN_DIR / "resultats_nilm.csv", index_col=[0, 1]).xs("Réseau", level="méthode")["F1"]
ecart = (s_ref.set_index("appareil")["F1"] - attendu).abs().max()
print("écart maximal avec les F1 de la semaine 6 :", round(float(ecart), 4), "->", "OK" if ecart < 1e-3 else "PROBLÈME : le modèle rechargé ne correspond pas")
s_ref.round(3)

## 4. Fenêtre glissante : ce que fera la carte
Toutes les 15 minutes, la carte prend **les 272 dernières minutes** de la courbe et lance le réseau. Le réseau a besoin de 127 minutes de « futur » pour un résultat complet : la carte garde donc le résultat des 15 minutes qui se terminent **127 minutes avant maintenant** (minutes 130 à 144 de la fenêtre). C'est ce quart d'heure-là qui part dans le message NB-IoT, d'où un retard de 127 à 142 minutes.

```
fenêtre de 272 min :  [ 127 min de passé | 15 min retenues | 127 min de « futur » ]
                                              ↑ résultat envoyé
```

Pour le vérifier, je découpe toute la période de test en fenêtres de 272 minutes espacées de 15 minutes, je garde à chaque fois les 15 minutes du milieu, et je recolle les morceaux : les scores doivent être ceux du modèle appliqué d'un seul tenant.

In [ ]:
SL = slice(WIN - RF - STEP, WIN - RF)                       # les 15 minutes retenues dans chaque fenêtre
ends = np.arange(i_test + RF + STEP, T + 1, STEP)           # fin de chaque fenêtre (une toutes les 15 min)
wins = np.stack([X[:, t - WIN:t] for t in ends])            # (nombre de fenêtres, 3, WIN)
with torch.no_grad():
    outs = np.concatenate([net(torch.tensor(wins[k:k + 1024])).numpy() for k in range(0, len(wins), 1024)])

def assemble(outs_):
    pred = np.zeros((n_app, T), np.float32); on = np.zeros((n_app, T), bool)
    for t, o in zip(ends, outs_):
        p_, o_ = to_pred(o[:, SL])
        pred[:, t - RF - STEP:t - RF], on[:, t - RF - STEP:t - RF] = p_, o_
    return pred, on

pred_win, on_win = assemble(outs)
s_win = scores(pred_win, on_win, "2. fenêtre glissante (PC)")
print(len(wins), "fenêtres de", WIN, "minutes sur la période de test")
s_win.round(3)

## 5. Export ONNX (le format lu par ST Edge AI)
- **Entrée** `x`, de forme (1, 3, 272) : [puissance totale en kW ; sinus de l'heure ; cosinus de l'heure] × 272 minutes.
- **Sortie** `y`, de forme (1, 8, 272) : 4 puissances normalisées (à multiplier par les échelles), puis 4 scores « allumé » (allumé si > 0).

J'utilise l'exportateur récent de PyTorch (*dynamo*) et, s'il n'est pas disponible, l'exportateur classique (opset 17). Le fichier est vérifié par `onnx.checker`, puis exécuté avec `onnxruntime` pour contrôler qu'il donne les mêmes sorties que PyTorch.

In [ ]:
FP32 = EXPORT_DIR / "nilm_fp32.onnx"
dummy = torch.tensor(wins[:1])
try:
    torch.onnx.export(net, (dummy,), str(FP32), input_names=["x"], output_names=["y"], dynamo=True, external_data=False)
    methode = "exportateur dynamo"
except Exception as e:
    print("exportateur dynamo indisponible (", type(e).__name__, ") -> exportateur classique")
    torch.onnx.export(net, (dummy,), str(FP32), input_names=["x"], output_names=["y"], opset_version=17, dynamo=False)
    methode = "exportateur classique"
m = onnx.load(str(FP32)); onnx.checker.check_model(m)
print(f"{FP32} écrit ({methode}, opset {m.opset_import[0].version}, {FP32.stat().st_size / 1024:.0f} Ko)")
print("entrée :", [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print("sortie :", [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])

sess32 = ort.InferenceSession(str(FP32), providers=["CPUExecutionProvider"])
IN, OUT = sess32.get_inputs()[0].name, sess32.get_outputs()[0].name
diff = max(np.abs(sess32.run([OUT], {IN: wins[k:k + 1]})[0][0] - outs[k]).max() for k in range(0, len(wins), 500))
print("écart maximal ONNX / PyTorch :", f"{diff:.2e}", "->", "OK" if diff < 1e-3 else "à regarder")

## 6. Compression en entiers 8 bits (int8)
Les poids et les calculs passent de nombres à virgule sur 32 bits à des entiers sur 8 bits (quantification statique d'`onnxruntime`). Pour choisir les plages de valeurs de chaque couche, l'outil observe le réseau sur des fenêtres réelles tirées **dans la période d'entraînement** (jamais dans le test), avec au moins 90 % de minutes valides. J'utilise le format **QDQ** avec des poids int8 **par canal** : c'est le format conseillé par ST pour l'accélérateur de la N6.

Point à retenir : la plage d'entrée apprise (pas d'environ 25 W) plafonne la puissance totale vers 5,4 kW ; elle sera à réajuster avec les mesures d'une maison française.

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType, CalibrationMethod
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
    PRE = EXPORT_DIR / "nilm_fp32_pre.onnx"
    quant_pre_process(str(FP32), str(PRE))
except Exception as e:
    print("pré-traitement ignoré :", type(e).__name__); PRE = FP32

ok_cal = [s for s in np.random.randint(0, i_val - WIN, 5 * N_CALIB) if valid[s:s + WIN].mean() > 0.9][:N_CALIB]
calib = np.stack([X[:, s:s + WIN] for s in ok_cal])

class Lecteur(CalibrationDataReader):
    def __init__(self, data): self.it = iter(data)
    def get_next(self):
        w = next(self.it, None)
        return None if w is None else {IN: w[None]}

INT8 = EXPORT_DIR / "nilm_int8.onnx"
t0 = time.time()
quantize_static(str(PRE), str(INT8), Lecteur(calib), quant_format=QuantFormat.QDQ, per_channel=True,
                activation_type=QuantType.QInt8, weight_type=QuantType.QInt8, calibrate_method=CalibrationMethod.MinMax)
print(f"{INT8} écrit ({INT8.stat().st_size / 1024:.0f} Ko, contre {FP32.stat().st_size / 1024:.0f} Ko en float32) en {time.time() - t0:.0f} s")

sess8 = ort.InferenceSession(str(INT8), providers=["CPUExecutionProvider"])
t0 = time.time()
outs8 = np.stack([sess8.run([OUT], {IN: w[None]})[0][0] for w in wins])
print(f"{len(wins)} fenêtres calculées en int8 en {time.time() - t0:.0f} s")
pred_8, on_8 = assemble(outs8)
s_8 = scores(pred_8, on_8, "3. int8 (ce qui ira sur la carte)")

## 7. Comparaison : la compression a-t-elle coûté quelque chose ?
Règle fixée avant de regarder : la version int8 est acceptée si le F1 ne baisse pas de plus de **0,02** sur chaque appareil (c'est l'ordre du bruit d'une graine à l'autre mesuré en semaine 6).

| Appareil | F1 PC | F1 fenêtre glissante | F1 int8 | Erreur d'énergie PC | Erreur d'énergie int8 |
|---|---|---|---|---|---|
| Lave-vaisselle | 0,871 | 0,871 | 0,877 | 14,8 % | 20,2 % |
| Lave-linge | 0,915 | 0,915 | 0,916 | 1,7 % | 5,3 % |
| Frigo | 0,810 | 0,810 | 0,799 | 15,8 % | 12,9 % |
| Bouilloire | 0,752 | 0,752 | 0,763 | 25,7 % | 18,6 % |

La fenêtre glissante redonne exactement les scores du modèle d'origine, et la version int8 reste à **±0,011** de F1 : elle est **acceptée**. Les erreurs d'énergie bougent de quelques points dans les deux sens, sans tendance défavorable.

In [ ]:
from IPython.display import display
cmp7 = pd.concat([s_ref, s_win, s_8]).pivot(index="appareil", columns="version")
display(cmp7.round(3))
perte = (cmp7[("F1", "1. modèle d'origine (PC)")] - cmp7[("F1", "3. int8 (ce qui ira sur la carte)")])
print("perte de F1 due à int8 + fenêtre :", perte.round(3).to_dict())
print("VERDICT :", "int8 ACCEPTÉ" if (perte <= 0.02).all() else "int8 À REVOIR (perte > 0,02 sur au moins un appareil)")

ax = cmp7["F1"].plot.bar(rot=0, ylim=(0, 1), title="F1 sur le test réel : PC, fenêtre glissante, int8")
ax.set_ylabel("F1"); ax.legend(fontsize=8, loc="lower right"); plt.tight_layout()
plt.savefig(EXPORT_DIR / "comparaison_int8.png", dpi=150); plt.show()
cmp7.round(4).to_csv(EXPORT_DIR / "comparaison_int8.csv")

## 8. Fichiers pour la carte
- `nilm_int8.onnx` (≈ 52 Ko) : le modèle à donner à ST Edge AI ;
- `nilm_meta.json` : tout ce que le programme de la carte doit savoir (ordre des entrées et sorties, échelles, fenêtre, pas, minutes retenues, seuils, calcul de l'énergie) ;
- `vecteurs_test.npz` + `vecteurs_test_*.csv` : 20 fenêtres réelles avec les sorties attendues. Une fois le modèle sur la carte, je lui donne ces fenêtres et je compare ses sorties à celles-ci ;
- `validation_entrees.npy` / `validation_sorties_int8.npy` : les mêmes données au format attendu par `stedgeai validate`.

In [ ]:
meta = {
    "modele": "nilm_int8.onnx", "run_source": str(RUN_DIR),
    "entree": {"nom": IN, "forme": [1, 3, WIN],
               "canaux": ["puissance totale (kW, moyenne sur 1 min)", "sin(2*pi*heure/24)", "cos(2*pi*heure/24)"],
               "minutes_manquantes": "mettre les 3 canaux à 0"},
    "sortie": {"nom": OUT, "forme": [1, 2 * n_app, WIN],
               "canaux_0_a_3": "puissance normalisée : W = max(y, 0) * echelle", "canaux_4_a_7": "allumé si y > 0",
               "appareils": APPLIANCES, "echelles_W": [float(v) for v in scale_y]},
    "fenetre_min": WIN, "pas_min": STEP, "contexte_min": RF,
    "minutes_retenues": [SL.start, SL.stop],
    "retard_min": [RF, RF + STEP],
    "seuils_allume_W": {a: float(ON_W[a]) for a in APPLIANCES},
    "energie_message": "Wh d'un appareil sur le quart d'heure = somme des 15 puissances (W) / 60",
}
(EXPORT_DIR / "nilm_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))

k_test = np.linspace(0, len(wins) - 1, 20).astype(int)
np.savez(EXPORT_DIR / "vecteurs_test.npz", x=wins[k_test], y_int8=outs8[k_test], y_fp32=outs[k_test], fin_fenetre=ends[k_test])
for j, k in enumerate(k_test[:3]):          # 3 fenêtres aussi en CSV, lisibles à la main
    pd.DataFrame(wins[k].T, columns=["kW", "sin_h", "cos_h"]).to_csv(EXPORT_DIR / f"vecteurs_test_{j}_entree.csv", index=False)
    pd.DataFrame(outs8[k].T, columns=[f"p_{a}" for a in APPLIANCES] + [f"on_{a}" for a in APPLIANCES]).to_csv(
        EXPORT_DIR / f"vecteurs_test_{j}_sortie_int8.csv", index=False)
print("fichiers dans", EXPORT_DIR.resolve(), ":")
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"   {p.name:32s} {p.stat().st_size / 1024:8.1f} Ko")

## 9. Étape suivante : du fichier ONNX à la carte
J'ai d'abord évalué le modèle avec **ST Edge AI Developer Cloud** (stedgeai-dc.st.com) : import de `nilm_int8.onnx`, étape *Optimize* avec les réglages par défaut, puis *Benchmark* sur une STM32N6 pour relever la RAM, la Flash, le temps d'un calcul et la répartition des couches entre l'accélérateur (NPU) et le processeur.

J'ai ensuite validé le modèle **sur ma propre carte NUCLEO-N657X0-Q**, en ligne de commande avec X-CUBE-AI 10.2 / ST Edge AI Core 2.2 (scripts et rapports dans `STM32/n6_validation/`) : compilation pour le NPU Neural-ART, écriture des poids dans la flash externe, puis `stedgeai validate --mode target` avec les 20 fenêtres de test, la carte étant démarrée en mode développement.

## 10. Conclusion : le modèle NILM embarqué

**Ce qui tourne sur la carte.** Le réseau de la semaine 6 (`ukdale_augoff_s0`) compressé en int8 : 22 120 paramètres, réseau convolutif 1D dilaté (3 entrées → 32 canaux, 7 blocs résiduels de dilatations 1 à 64 min → 8 sorties).

**Résultats mesurés sur la NUCLEO-N657X0-Q :**

| Mesure | Résultat |
|---|---|
| Temps d'une inférence (fenêtre de 272 min) | **2,05 ms** (≈ 489 inférences/s possibles) |
| Besoin réel | 1 inférence toutes les 15 minutes |
| Étapes exécutées par le NPU Neural-ART | 38 sur 54 (les convolutions) ; les 16 autres sont des réorganisations de données confiées au Cortex-M55 |
| Mémoire | 23 Ko de poids (flash externe), ≈ 60 Ko de RAM d'activations (npuRAM5) |
| Accord carte / PC sur 20 fenêtres réelles | cosinus 0,9997 ; 99,5 % de décisions allumé/éteint identiques |
| Effet de la compression int8 sur le F1 | ±0,011 au plus |

**Entrée sur la carte** : toutes les 15 minutes, un tableau 3 × 272 = les 272 dernières minutes de puissance totale (kW, moyenne par minute) + sinus et cosinus de l'heure. Les minutes manquantes sont mises à 0 sur les 3 canaux.

**Sortie du réseau** : un tableau 8 × 272. La carte n'en garde que les minutes 130 à 144 (le quart d'heure terminé il y a 127 minutes), convertit les 4 premières lignes en watts (× 249,7 / 2 405,5 / 1 975,3 / 2 392,1 W), met à 0 les appareils dont le score est négatif, puis calcule l'énergie de chaque appareil sur le quart d'heure (somme des 15 puissances / 60, en Wh).

**Message envoyé vers Home Assistant** (proposition à valider avec les parties A et B) : topic `maison/nilm_01/q`, contenu `hhmm;frigo;bouilloire;lave_linge;lave_vaisselle;total` en Wh, par exemple `1615;21;48;0;0;310` (18 caractères, sous la limite de 50 caractères du module ST87M01), soit 96 messages par jour.

**Intérêt.**
- **IA en périphérie (edge AI)** : la désagrégation est faite dans le capteur ; la courbe brute ne quitte pas la maison, seules 5 énergies par quart d'heure partent sur le lien NB-IoT. C'est plus sobre en données et meilleur pour la confidentialité.
- **Coût de calcul négligeable** : 2 ms toutes les 15 minutes, ce qui laisse le microcontrôleur disponible pour la mesure et la communication et permet une faible consommation.
- Home Assistant reçoit directement une consommation par appareil, utilisable dans le tableau Énergie et pour planifier la flexibilité.

**Limites et suite.**
- Validé avec des fenêtres enregistrées : il reste à brancher le vrai capteur (pince de courant, éventuellement croisée avec la TIC du Linky) et l'envoi NB-IoT du kit ST87M01.
- Entrée plafonnée vers 5,4 kW par la quantification, et modèle appris sur une maison britannique : quelques jours de mesures réelles serviront à réajuster la plage d'entrée et, si besoin, à affiner le modèle.
- Retard d'environ 2 h entre la consommation et le message : adapté au suivi et à la planification, pas au pilotage instantané.